ANÁLISIS DE DESEMPEÑO EDITORIAL - MILENIO.COM
Segmentación de Editores con Machine Learning (KMeans)

INSTALACIÓN Y CARGA DE LIBRERÍAS

In [ ]:
!pip install -q plotly kaleido scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy import stats

# Visualización
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

# Utilidades
from datetime import datetime
from collections import Counter
import json

print("✅ Todas las librerías cargadas correctamente")
print(f"📅 Fecha de análisis: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


✅ Todas las librerías cargadas correctamente
📅 Fecha de análisis: 2026-02-06 20:46


Carga de datos

In [ ]:
# Opción 1: Cargar desde archivo local en Colab
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Leer el CSV con el encoding correcto
df = pd.read_csv(filename, encoding='latin-1')

# Convertir la columna 'Fecha' a formato de fecha
df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce')

# Opción 2: Si ya tienes el archivo en Drive (comentar la opción 1 y descomentar esta)
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/Tematica.csv', encoding='latin-1')

print(f"✅ Dataset cargado: {df.shape[0]:,} filas, {df.shape[1]} columnas")
print(f"📅 Rango de fechas: {df['Fecha'].min()} a {df['Fecha'].max()}")

Saving Tematica.csv to Tematica (1).csv
✅ Dataset cargado: 47,064 filas, 26 columnas
📅 Rango de fechas: 2025-10-01 00:00:00 a 2026-01-29 00:00:00


Limpieza y preparación de datos

In [ ]:
print("\n" + "="*80)
print("🧹 FASE 1: LIMPIEZA DE DATOS")
print("="*80)

# Renombrar columnas para facilitar manejo
df = df.rename(columns={
    'editor': 'Editor',
    'Pv´s': 'PVs',
    'Ads Por Página': 'AdsPorPagina'
})

# Mostrar información inicial
print(f"\n📊 Dataset original: {df.shape[0]:,} notas")
print(f"👥 Editores únicos: {df['Editor'].nunique()}")
print(f"✍️  Autores únicos: {df['Autor'].nunique()}")

# 1. ELIMINAR FILAS CON VALORES NULOS EN MÉTRICAS CLAVE
df_clean = df.dropna(subset=['Editor', 'Autor', 'Registros', 'PVs', 'Scroll', 'RFV', 'AdsPorPagina'])
print(f"✅ Después de eliminar nulos: {df_clean.shape[0]:,} notas ({(1 - df_clean.shape[0]/df.shape[0])*100:.1f}% eliminado)")

# 2. NORMALIZAR NOMBRES DE EDITORES Y AUTORES
df_clean['Editor'] = df_clean['Editor'].str.strip().str.lower()
df_clean['Autor'] = df_clean['Autor'].str.strip().str.lower()

# 3. CONTAR NOTAS POR EDITOR
notas_por_editor = df_clean.groupby('Editor').size().reset_index(name='num_notas')

# 4. EXCLUIR EDITORES CON MENOS DE 12 NOTAS
editores_validos = notas_por_editor[notas_por_editor['num_notas'] >= 12]['Editor'].tolist()
df_clean = df_clean[df_clean['Editor'].isin(editores_validos)]
print(f"✅ Editores con 12+ notas: {len(editores_validos)} editores")
print(f"✅ Notas válidas: {df_clean.shape[0]:,}")

# 5. EXCLUIR EDITOR ESPECÍFICO (miriam.castro)
if 'miriam.castro' in df_clean['Editor'].values:
    df_clean = df_clean[df_clean['Editor'] != 'miriam.castro']
    print("✅ Editor 'miriam.castro' excluido (dato atípico)")

# 6. CONVERTIR MÉTRICAS A NUMÉRICO
metricas_numericas = ['Registros', 'PVs', 'Scroll', 'RFV', 'AdsPorPagina']
for col in metricas_numericas:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Eliminar filas con valores nulos resultantes
df_clean = df_clean.dropna(subset=metricas_numericas)

print(f"\n✅ Dataset limpio final: {df_clean.shape[0]:,} notas, {df_clean['Editor'].nunique()} editores")


🧹 FASE 1: LIMPIEZA DE DATOS

📊 Dataset original: 47,064 notas
👥 Editores únicos: 140
✍️  Autores únicos: 1296
✅ Después de eliminar nulos: 45,882 notas (2.5% eliminado)
✅ Editores con 12+ notas: 117 editores
✅ Notas válidas: 45,812

✅ Dataset limpio final: 45,812 notas, 117 editores


AGREGACIÓN POR EDITOR

In [ ]:
print("\n" + "="*80)
print("📈 FASE 2: AGREGACIÓN DE MÉTRICAS POR EDITOR")
print("="*80)

# Agregación básica de métricas
metricas_editor = df_clean.groupby('Editor').agg({
    'Registros': 'sum',
    'PVs': 'sum',
    'Scroll': 'mean',
    'RFV': 'mean',
    'AdsPorPagina': 'mean',
    'id': 'count'
}).reset_index()

metricas_editor = metricas_editor.rename(columns={'id': 'num_notas'})

# FEATURE ENGINEERING: Métricas derivadas
print("\n🔧 Creando features derivadas...")

# 1. EFICIENCIA REGISTROS POR PV
metricas_editor['eficiencia_registros_pv'] = np.where(
    metricas_editor['PVs'] > 0,
    metricas_editor['Registros'] / metricas_editor['PVs'] * 1000,  # Por cada 1000 PVs
    0
)

# 2. CONSISTENCIA EDITORIAL (Coeficiente de variación de Registros)
consistencia = df_clean.groupby('Editor')['Registros'].agg(['std', 'mean']).reset_index()
consistencia['consistencia_editorial'] = np.where(
    consistencia['mean'] > 0,
    1 - (consistencia['std'] / consistencia['mean']),  # Mayor valor = más consistente
    0
)
consistencia['consistencia_editorial'] = consistencia['consistencia_editorial'].clip(lower=0)
metricas_editor = metricas_editor.merge(consistencia[['Editor', 'consistencia_editorial']], on='Editor')

# 3. ÍNDICE DE ORIGINALIDAD (% de notas donde Editor == Autor)
originalidad = df_clean.copy()
originalidad['es_original'] = (originalidad['Editor'] == originalidad['Autor']).astype(int)
indice_originalidad = originalidad.groupby('Editor')['es_original'].mean().reset_index()
indice_originalidad.columns = ['Editor', 'indice_originalidad']
metricas_editor = metricas_editor.merge(indice_originalidad, on='Editor')

# 4. DIVERSIDAD TEMÁTICA (Entropía de Shannon)
def calcular_entropia(temas):
    """Calcula la entropía de Shannon para medir diversidad temática"""
    if len(temas) == 0:
        return 0
    conteo = Counter(temas)
    total = len(temas)
    entropia = -sum((count/total) * np.log2(count/total) for count in conteo.values() if count > 0)
    return entropia

diversidad = df_clean.groupby('Editor')['tema'].apply(lambda x: calcular_entropia(x.dropna())).reset_index()
diversidad.columns = ['Editor', 'diversidad_tematica']
metricas_editor = metricas_editor.merge(diversidad, on='Editor')

# 5. SCORE GLOBAL PONDERADO
"""
PONDERACIÓN DEL SCORE GLOBAL:
- Registros: 40% (peso más alto - objetivo principal de negocio)
- PVs: 20% (volumen de audiencia)
- RFV: 15% (engagement)
- Scroll: 15% (consumo de contenido)
- Eficiencia Registros/PV: 10% (productividad)
"""

# Normalizar cada métrica a escala 0-100
def normalizar_0_100(serie):
    """Normaliza una serie a escala 0-100"""
    if serie.max() == serie.min():
        return pd.Series([50] * len(serie), index=serie.index)
    return ((serie - serie.min()) / (serie.max() - serie.min())) * 100

metricas_editor['registros_norm'] = normalizar_0_100(metricas_editor['Registros'])
metricas_editor['pvs_norm'] = normalizar_0_100(metricas_editor['PVs'])
metricas_editor['rfv_norm'] = normalizar_0_100(metricas_editor['RFV'])
metricas_editor['scroll_norm'] = normalizar_0_100(metricas_editor['Scroll'])
metricas_editor['eficiencia_norm'] = normalizar_0_100(metricas_editor['eficiencia_registros_pv'])

# Calcular score global ponderado
metricas_editor['score_global'] = (
    metricas_editor['registros_norm'] * 0.40 +
    metricas_editor['pvs_norm'] * 0.20 +
    metricas_editor['rfv_norm'] * 0.15 +
    metricas_editor['scroll_norm'] * 0.15 +
    metricas_editor['eficiencia_norm'] * 0.10
)

print(f"✅ Features derivados creados para {len(metricas_editor)} editores")
print("\nMétricas disponibles:")
print("  • Registros (suma)")
print("  • PVs (suma)")
print("  • Scroll (promedio)")
print("  • RFV (promedio)")
print("  • AdsPorPagina (promedio)")
print("  • eficiencia_registros_pv (Registros por cada 1000 PVs)")
print("  • consistencia_editorial (estabilidad de desempeño)")
print("  • indice_originalidad (% notas propias)")
print("  • diversidad_tematica (entropía de Shannon)")
print("  • score_global (ponderado con énfasis en Registros)")



📈 FASE 2: AGREGACIÓN DE MÉTRICAS POR EDITOR

🔧 Creando features derivadas...
✅ Features derivados creados para 117 editores

Métricas disponibles:
  • Registros (suma)
  • PVs (suma)
  • Scroll (promedio)
  • RFV (promedio)
  • AdsPorPagina (promedio)
  • eficiencia_registros_pv (Registros por cada 1000 PVs)
  • consistencia_editorial (estabilidad de desempeño)
  • indice_originalidad (% notas propias)
  • diversidad_tematica (entropía de Shannon)
  • score_global (ponderado con énfasis en Registros)


Tratamiento de Outliers

In [ ]:
print("\n" + "="*80)
print("🔍 FASE 3: TRATAMIENTO DE OUTLIERS")
print("="*80)

# Identificar outliers usando método IQR y winsorization
def winsorize_column(df, col, lower_percentile=5, upper_percentile=95):
    """Aplica winsorization a una columna (clip extremos)"""
    lower = df[col].quantile(lower_percentile / 100)
    upper = df[col].quantile(upper_percentile / 100)
    df[col + '_winsorized'] = df[col].clip(lower=lower, upper=upper)
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    return outliers

# Aplicar winsorization a métricas principales
features_para_clustering = ['Registros', 'PVs', 'Scroll', 'RFV', 'eficiencia_registros_pv', 'score_global']

print("\nAplicando winsorization (percentiles 5-95):")
for feature in features_para_clustering:
    n_outliers = winsorize_column(metricas_editor, feature)
    print(f"  • {feature}: {n_outliers} outliers tratados")

# Crear dataframe para clustering con valores winsorized
features_winsorized = [f + '_winsorized' for f in features_para_clustering]
df_clustering = metricas_editor[['Editor'] + features_winsorized].copy()
df_clustering.columns = ['Editor'] + features_para_clustering  # Renombrar sin _winsorized

print("\n✅ Outliers tratados exitosamente")




🔍 FASE 3: TRATAMIENTO DE OUTLIERS

Aplicando winsorization (percentiles 5-95):
  • Registros: 6 outliers tratados
  • PVs: 12 outliers tratados
  • Scroll: 12 outliers tratados
  • RFV: 12 outliers tratados
  • eficiencia_registros_pv: 6 outliers tratados
  • score_global: 12 outliers tratados

✅ Outliers tratados exitosamente


CLUSTERING CON KMEANS

In [ ]:
print("\n" + "="*80)
print("🤖 FASE 4: CLUSTERING CON KMEANS")
print("="*80)

# Preparar features para clustering
X = df_clustering[features_para_clustering].values

# Escalado con RobustScaler (más robusto a outliers que StandardScaler)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# MÉTODO DEL CODO + SILHOUETTE SCORE
print("\n🔍 Determinando número óptimo de clusters...")

inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Visualizar método del codo
fig_elbow = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Método del Codo', 'Silhouette Score')
)

fig_elbow.add_trace(
    go.Scatter(x=list(K_range), y=inertias, mode='lines+markers', name='Inercia'),
    row=1, col=1
)

fig_elbow.add_trace(
    go.Scatter(x=list(K_range), y=silhouette_scores, mode='lines+markers',
               name='Silhouette', line=dict(color='orange')),
    row=1, col=2
)

fig_elbow.update_xaxes(title_text="Número de Clusters (K)", row=1, col=1)
fig_elbow.update_xaxes(title_text="Número de Clusters (K)", row=1, col=2)
fig_elbow.update_yaxes(title_text="Inercia", row=1, col=1)
fig_elbow.update_yaxes(title_text="Silhouette Score", row=1, col=2)
fig_elbow.update_layout(height=400, showlegend=False, title_text="Análisis de K óptimo")
fig_elbow.show()

# Seleccionar K óptimo (punto donde mejora marginal disminuye + buen silhouette)
# Por default, seleccionamos 4 clusters para las etiquetas narrativas
K_OPTIMO = 4

print(f"\n✅ Número de clusters seleccionado: {K_OPTIMO}")
print(f"   Silhouette Score para K={K_OPTIMO}: {silhouette_scores[K_OPTIMO-2]:.3f}")

# Entrenar modelo final con K óptimo
kmeans_final = KMeans(n_clusters=K_OPTIMO, random_state=42, n_init=10)
df_clustering['cluster'] = kmeans_final.fit_predict(X_scaled)

print(f"\n📊 Distribución de editores por cluster:")
print(df_clustering['cluster'].value_counts().sort_index())


🤖 FASE 4: CLUSTERING CON KMEANS

🔍 Determinando número óptimo de clusters...



✅ Número de clusters seleccionado: 4
   Silhouette Score para K=4: 0.398

📊 Distribución de editores por cluster:
cluster
0    13
1    60
2    31
3    13
Name: count, dtype: int64


ETIQUETADO NARRATIVO DE CLUSTERS

In [ ]:
print("\n" + "="*80)
print("👑 FASE 5: ETIQUETADO NARRATIVO DE CLUSTERS")
print("="*80)

# Calcular métricas promedio por cluster
cluster_stats = df_clustering.groupby('cluster').agg({
    'Registros': 'mean',
    'PVs': 'mean',
    'score_global': 'mean',
    'Editor': 'count'
}).reset_index()
cluster_stats.columns = ['cluster', 'Registros_promedio', 'PVs_promedio', 'score_global_promedio', 'num_editores']

# Ordenar clusters por score_global (de mayor a menor)
cluster_stats = cluster_stats.sort_values('score_global_promedio', ascending=False).reset_index(drop=True)

# Mapear etiquetas narrativas
etiquetas_narrativas = ['Reyes y Reinas', 'Príncipes y Princesas', 'Duques y Duquesas', 'Sapitos y Sapitas']

# Crear mapping de cluster original a etiqueta
cluster_to_label = dict(zip(cluster_stats['cluster'], etiquetas_narrativas[:K_OPTIMO]))

# Aplicar etiquetas
df_clustering['etiqueta'] = df_clustering['cluster'].map(cluster_to_label)

# Merge con metricas_editor
metricas_editor = metricas_editor.merge(df_clustering[['Editor', 'cluster', 'etiqueta']], on='Editor')

print("\n🏆 MAPPING DE CLUSTERS A ETIQUETAS:")
print("="*50)
for idx, row in cluster_stats.iterrows():
    etiqueta = etiquetas_narrativas[idx]
    print(f"\n{etiqueta}:")
    print(f"  • Editores: {int(row['num_editores'])}")
    print(f"  • Score Global: {row['score_global_promedio']:.1f}")
    print(f"  • Registros promedio: {int(row['Registros_promedio']):,}")
    print(f"  • PVs promedio: {int(row['PVs_promedio']):,}")


👑 FASE 5: ETIQUETADO NARRATIVO DE CLUSTERS

🏆 MAPPING DE CLUSTERS A ETIQUETAS:

Reyes y Reinas:
  • Editores: 13
  • Score Global: 34.4
  • Registros promedio: 2,655
  • PVs promedio: 3,523,171

Príncipes y Princesas:
  • Editores: 31
  • Score Global: 22.9
  • Registros promedio: 914
  • PVs promedio: 1,304,421

Duques y Duquesas:
  • Editores: 13
  • Score Global: 8.8
  • Registros promedio: 206
  • PVs promedio: 145,173

Sapitos y Sapitas:
  • Editores: 60
  • Score Global: 7.7
  • Registros promedio: 140
  • PVs promedio: 349,214


PCA PARA VISUALIZACIÓN

In [ ]:
print("\n" + "="*80)
print("🎨 FASE 6: REDUCCIÓN DIMENSIONAL CON PCA")
print("="*80)

# Aplicar PCA para reducir a 2 dimensiones
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Agregar coordenadas PCA al dataframe
df_clustering['pca_1'] = X_pca[:, 0]
df_clustering['pca_2'] = X_pca[:, 1]

# Merge con metricas_editor
metricas_editor = metricas_editor.merge(df_clustering[['Editor', 'pca_1', 'pca_2']], on='Editor')

print(f"✅ PCA completado")
print(f"   Varianza explicada PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"   Varianza explicada PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"   Varianza total explicada: {sum(pca.explained_variance_ratio_)*100:.1f}%")


🎨 FASE 6: REDUCCIÓN DIMENSIONAL CON PCA
✅ PCA completado
   Varianza explicada PC1: 64.8%
   Varianza explicada PC2: 17.0%
   Varianza total explicada: 81.8%


ANÁLISIS TEMÁTICO

In [ ]:
print("\n" + "="*80)
print("📚 FASE 7: ANÁLISIS TEMÁTICO POR EDITOR")
print("="*80)

# Top temas por editor
def obtener_top_temas(df, editor, top_n=5):
    """Obtiene los top N temas de un editor con porcentajes"""
    notas_editor = df[df['Editor'] == editor]
    temas_count = notas_editor['tema'].value_counts()
    total_notas = len(notas_editor)

    top_temas = []
    for tema, count in temas_count.head(top_n).items():
        porcentaje = (count / total_notas) * 100
        top_temas.append({
            'tema': tema,
            'num_notas': count,
            'porcentaje': porcentaje
        })

    return top_temas

# Top entidades (personaje_principal) por editor
def obtener_top_entidades(df, editor, top_n=10):
    """Obtiene las top N entidades de un editor"""
    notas_editor = df[df['Editor'] == editor]
    entidades_count = notas_editor['personaje_principal'].dropna().value_counts()

    top_entidades = []
    for entidad, count in entidades_count.head(top_n).items():
        top_entidades.append({
            'entidad': entidad,
            'num_notas': count
        })

    return top_entidades

print("\n🔍 Generando análisis temático por editor...")

# Crear diccionarios de temas y entidades por editor
temas_por_editor = {}
entidades_por_editor = {}

for editor in metricas_editor['Editor']:
    temas_por_editor[editor] = obtener_top_temas(df_clean, editor, top_n=5)
    entidades_por_editor[editor] = obtener_top_entidades(df_clean, editor, top_n=10)

print(f"✅ Análisis temático completado para {len(metricas_editor)} editores")

# Análisis temático por CLUSTER
print("\n📊 Generando análisis temático por cluster...")

cluster_temas = {}
for etiqueta in etiquetas_narrativas[:K_OPTIMO]:
    editores_cluster = metricas_editor[metricas_editor['etiqueta'] == etiqueta]['Editor'].tolist()
    notas_cluster = df_clean[df_clean['Editor'].isin(editores_cluster)]
    temas_count = notas_cluster['tema'].value_counts().head(10)
    cluster_temas[etiqueta] = temas_count.to_dict()

print("✅ Análisis por cluster completado")



📚 FASE 7: ANÁLISIS TEMÁTICO POR EDITOR

🔍 Generando análisis temático por editor...
✅ Análisis temático completado para 117 editores

📊 Generando análisis temático por cluster...
✅ Análisis por cluster completado


DASHBOARD INTERACTIVO

In [ ]:
print("\n" + "="*80)
print("📊 FASE 8: GENERACIÓN DE DASHBOARD INTERACTIVO")
print("="*80)

# Configuración de tema
pio.templates.default = "plotly_white"

# Colores por etiqueta
colores_etiquetas = {
    'Reyes y Reinas': '#FFD700',      # Dorado
    'Príncipes y Princesas': '#C0C0C0', # Plateado
    'Duques y Duquesas': '#CD7F32',    # Bronce
    'Sapitos y Sapitas': '#90EE90'     # Verde claro
}

# ═══════════════════════════════════════════════════════════════════════════
# GRÁFICA 1: KPIs PRINCIPALES (SIN DECIMALES)
# ═══════════════════════════════════════════════════════════════════════════

total_editores = int(len(metricas_editor))
total_notas = int(metricas_editor['num_notas'].sum())
total_pvs = int(metricas_editor['PVs'].sum())
total_registros = int(metricas_editor['Registros'].sum())
score_promedio = int(metricas_editor['score_global'].mean())

kpis_html = f"""
<div style='display: flex; justify-content: space-around; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 30px; border-radius: 15px; margin-bottom: 30px; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
    <div style='text-align: center; color: white;'>
        <div style='font-size: 48px; font-weight: bold;'>{total_editores}</div>
        <div style='font-size: 16px; margin-top: 10px;'>👥 Editores</div>
    </div>
    <div style='text-align: center; color: white;'>
        <div style='font-size: 48px; font-weight: bold;'>{total_notas:,}</div>
        <div style='font-size: 16px; margin-top: 10px;'>📝 Notas</div>
    </div>
    <div style='text-align: center; color: white;'>
        <div style='font-size: 48px; font-weight: bold;'>{total_pvs:,}</div>
        <div style='font-size: 16px; margin-top: 10px;'>👁️ PVs Totales</div>
    </div>
    <div style='text-align: center; color: white;'>
        <div style='font-size: 48px; font-weight: bold;'>{total_registros:,}</div>
        <div style='font-size: 16px; margin-top: 10px;'>🎯 Registros Totales</div>
    </div>
    <div style='text-align: center; color: white;'>
        <div style='font-size: 48px; font-weight: bold;'>{score_promedio}</div>
        <div style='font-size: 16px; margin-top: 10px;'>⭐ Score Promedio</div>
    </div>
</div>
"""

# ═══════════════════════════════════════════════════════════════════════════
# GRÁFICA 2: SCATTER PCA INTERACTIVO
# ═══════════════════════════════════════════════════════════════════════════

fig_pca = go.Figure()

for etiqueta in etiquetas_narrativas[:K_OPTIMO]:
    df_etiqueta = metricas_editor[metricas_editor['etiqueta'] == etiqueta]

    fig_pca.add_trace(go.Scatter(
        x=df_etiqueta['pca_1'],
        y=df_etiqueta['pca_2'],
        mode='markers',
        name=etiqueta,
        marker=dict(
            size=df_etiqueta['Registros'] / df_etiqueta['Registros'].max() * 50 + 10,
            color=colores_etiquetas[etiqueta],
            line=dict(width=2, color='white'),
            opacity=0.8
        ),
        text=df_etiqueta['Editor'],
        customdata=df_etiqueta[['Registros', 'PVs', 'score_global', 'num_notas']],
        hovertemplate=
        '<b>%{text}</b><br>' +
        '<b>Etiqueta:</b> ' + etiqueta + '<br>' +
        '<b>Registros:</b> %{customdata[0]:,.0f}<br>' +
        '<b>PVs:</b> %{customdata[1]:,.0f}<br>' +
        '<b>Score Global:</b> %{customdata[2]:.1f}<br>' +
        '<b>Notas:</b> %{customdata[3]:,.0f}<br>' +
        '<extra></extra>'
    ))

fig_pca.update_layout(
    title='Mapa de Editores por Desempeño (PCA)',
    xaxis_title=f'Componente Principal 1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)',
    yaxis_title=f'Componente Principal 2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)',
    height=600,
    hovermode='closest',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# ═══════════════════════════════════════════════════════════════════════════
# GRÁFICA 3: TOP 10 EDITORES
# ═══════════════════════════════════════════════════════════════════════════

top_10_editores = metricas_editor.nlargest(10, 'score_global')

fig_top10 = go.Figure()

# Barras de Score Global
fig_top10.add_trace(go.Bar(
    y=top_10_editores['Editor'],
    x=top_10_editores['score_global'],
    name='Score Global',
    orientation='h',
    marker=dict(color='#667eea'),
    text=top_10_editores['score_global'].round(1),
    textposition='outside'
))

# Barras de Registros (eje secundario)
fig_top10.add_trace(go.Bar(
    y=top_10_editores['Editor'],
    x=top_10_editores['Registros'],
    name='Registros',
    orientation='h',
    marker=dict(color='#f093fb'),
    text=top_10_editores['Registros'].astype(int),
    textposition='outside',
    yaxis='y2',
    xaxis='x2'
))

fig_top10.update_layout(
    title='Top 10 Editores por Score Global y Registros',
    height=500,
    xaxis=dict(title='Score Global'),
    xaxis2=dict(title='Registros', overlaying='x', side='top'),
    yaxis=dict(title=''),
    yaxis2=dict(overlaying='y', side='left'),
    barmode='group',
    showlegend=True
)

# ═══════════════════════════════════════════════════════════════════════════
# GRÁFICA 4: DISTRIBUCIÓN DE ETIQUETAS
# ═══════════════════════════════════════════════════════════════════════════

etiquetas_dist = metricas_editor.groupby('etiqueta').agg({
    'Editor': 'count',
    'Registros': 'sum'
}).reset_index()
etiquetas_dist.columns = ['etiqueta', 'num_editores', 'registros_totales']

# Ordenar por etiqueta (según orden narrativo)
etiquetas_orden = {etiqueta: i for i, etiqueta in enumerate(etiquetas_narrativas[:K_OPTIMO])}
etiquetas_dist['orden'] = etiquetas_dist['etiqueta'].map(etiquetas_orden)
etiquetas_dist = etiquetas_dist.sort_values('orden')

fig_dist = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Conteo de Editores', 'Registros Totales'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}]]
)

# Conteo de editores
fig_dist.add_trace(
    go.Bar(
        x=etiquetas_dist['etiqueta'],
        y=etiquetas_dist['num_editores'],
        marker=dict(color=[colores_etiquetas[e] for e in etiquetas_dist['etiqueta']]),
        text=etiquetas_dist['num_editores'],
        textposition='outside',
        showlegend=False
    ),
    row=1, col=1
)

# Registros totales
fig_dist.add_trace(
    go.Bar(
        x=etiquetas_dist['etiqueta'],
        y=etiquetas_dist['registros_totales'],
        marker=dict(color=[colores_etiquetas[e] for e in etiquetas_dist['etiqueta']]),
        text=etiquetas_dist['registros_totales'].astype(int),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

fig_dist.update_layout(height=400, title_text='Distribución por Etiqueta')
fig_dist.update_xaxes(tickangle=-45)

# ═══════════════════════════════════════════════════════════════════════════
# GRÁFICA 5: HEATMAP CLUSTER VS TEMA
# ═══════════════════════════════════════════════════════════════════════════

# Preparar datos para heatmap
heatmap_data = []
temas_unicos = set()

for etiqueta, temas in cluster_temas.items():
    temas_unicos.update(temas.keys())

temas_list = sorted(list(temas_unicos))[:15]  # Top 15 temas

for etiqueta in etiquetas_narrativas[:K_OPTIMO]:
    row = []
    for tema in temas_list:
        row.append(cluster_temas[etiqueta].get(tema, 0))
    heatmap_data.append(row)

fig_heatmap = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=[tema[:40] + '...' if len(tema) > 40 else tema for tema in temas_list],
    y=etiquetas_narrativas[:K_OPTIMO],
    colorscale='Blues',
    text=heatmap_data,
    texttemplate='%{text}',
    textfont={"size": 10},
    hovertemplate='<b>%{y}</b><br>%{x}<br>Notas: %{z}<extra></extra>'
))

fig_heatmap.update_layout(
    title='Temas Dominantes por Cluster',
    height=400,
    xaxis_title='Temas',
    yaxis_title='Cluster'
)
fig_heatmap.update_xaxes(tickangle=-45)

# ═══════════════════════════════════════════════════════════════════════════
# GRÁFICA 6: RANKING ÍNDICE DE ORIGINALIDAD
# ═══════════════════════════════════════════════════════════════════════════

top_originalidad = metricas_editor.nlargest(15, 'indice_originalidad')

fig_originalidad = go.Figure(go.Bar(
    x=top_originalidad['indice_originalidad'] * 100,
    y=top_originalidad['Editor'],
    orientation='h',
    marker=dict(
        color=top_originalidad['indice_originalidad'] * 100,
        colorscale='Greens',
        showscale=True,
        colorbar=dict(title='%')
    ),
    text=top_originalidad['indice_originalidad'].apply(lambda x: f'{x*100:.1f}%'),
    textposition='outside'
))

fig_originalidad.update_layout(
    title='Top 15 Editores por Índice de Originalidad (% Notas Propias)',
    xaxis_title='% de Notas Donde Editor = Autor',
    yaxis_title='',
    height=500
)


📊 FASE 8: GENERACIÓN DE DASHBOARD INTERACTIVO


COMPILAR DASHBOARD HTML

In [ ]:
html_content = f"""
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dashboard Editorial - Milenio.com</title>
    <style>
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            margin: 0;
            padding: 20px;
            background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
        }}
        .container {{
            max-width: 1400px;
            margin: 0 auto;
            background: white;
            padding: 30px;
            border-radius: 20px;
            box-shadow: 0 10px 40px rgba(0,0,0,0.1);
        }}
        h1 {{
            text-align: center;
            color: #2d3748;
            font-size: 36px;
            margin-bottom: 10px;
        }}
        .subtitle {{
            text-align: center;
            color: #718096;
            font-size: 18px;
            margin-bottom: 40px;
        }}
        .section {{
            margin-bottom: 40px;
        }}
        .section-title {{
            font-size: 24px;
            color: #2d3748;
            margin-bottom: 20px;
            border-left: 5px solid #667eea;
            padding-left: 15px;
        }}
        .chart {{
            margin-bottom: 30px;
        }}
        .footer {{
            text-align: center;
            color: #a0aec0;
            font-size: 14px;
            margin-top: 50px;
            padding-top: 20px;
            border-top: 2px solid #e2e8f0;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>📊 Dashboard de Desempeño Editorial</h1>
        <div class="subtitle">Milenio.com - Análisis de Editores con Machine Learning</div>

        <!-- KPIs -->
        <div class="section">
            {kpis_html}
        </div>

        <!-- Scatter PCA -->
        <div class="section">
            <div class="section-title">🗺️ Mapa de Editores (Análisis PCA)</div>
            <div class="chart">
                {pio.to_html(fig_pca, include_plotlyjs='cdn', full_html=False)}
            </div>
        </div>

        <!-- Top 10 Editores -->
        <div class="section">
            <div class="section-title">🏆 Top 10 Editores</div>
            <div class="chart">
                {pio.to_html(fig_top10, include_plotlyjs=False, full_html=False)}
            </div>
        </div>

        <!-- Distribución por Etiqueta -->
        <div class="section">
            <div class="section-title">👥 Distribución por Etiqueta</div>
            <div class="chart">
                {pio.to_html(fig_dist, include_plotlyjs=False, full_html=False)}
            </div>
        </div>

        <!-- Heatmap Temas -->
        <div class="section">
            <div class="section-title">📚 Temas por Cluster</div>
            <div class="chart">
                {pio.to_html(fig_heatmap, include_plotlyjs=False, full_html=False)}
            </div>
        </div>

        <!-- Índice de Originalidad -->
        <div class="section">
            <div class="section-title">✍️ Índice de Originalidad Editorial</div>
            <div class="chart">
                {pio.to_html(fig_originalidad, include_plotlyjs=False, full_html=False)}
            </div>
        </div>

        <!-- Footer -->
        <div class="footer">
            <p>📅 Generado el: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
            <p>🔬 Metodología: KMeans Clustering + PCA | 🎯 Ponderación: Registros (40%), PVs (20%), RFV (15%), Scroll (15%), Eficiencia (10%)</p>
            <p>📊 Grupo Multimedios - Equipo de Data Science</p>
        </div>
    </div>
</body>
</html>
"""

# Guardar dashboard
with open('dashboard_editorial_milenio.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("\n✅ Dashboard generado: 'dashboard_editorial_milenio.html'")


✅ Dashboard generado: 'dashboard_editorial_milenio.html'


EXPORTAR RESULTADOS A CSV

In [ ]:
print("\n" + "="*80)
print("💾 FASE 9: EXPORTACIÓN DE RESULTADOS")
print("="*80)

# CSV 1: Resultados por editor
resultados_editores = metricas_editor[[
    'Editor', 'etiqueta', 'cluster', 'num_notas', 'Registros', 'PVs',
    'Scroll', 'RFV', 'AdsPorPagina', 'score_global', 'eficiencia_registros_pv',
    'consistencia_editorial', 'indice_originalidad', 'diversidad_tematica'
]].copy()

resultados_editores = resultados_editores.sort_values('score_global', ascending=False)
resultados_editores.to_csv('resultados_por_editor.csv', index=False, encoding='utf-8-sig')
print("✅ Exportado: 'resultados_por_editor.csv'")

# CSV 2: Resumen por cluster
resumen_cluster = metricas_editor.groupby('etiqueta').agg({
    'Editor': 'count',
    'num_notas': 'sum',
    'Registros': ['sum', 'mean'],
    'PVs': ['sum', 'mean'],
    'score_global': 'mean',
    'indice_originalidad': 'mean',
    'diversidad_tematica': 'mean'
}).reset_index()

resumen_cluster.columns = [
    'Etiqueta', 'Num_Editores', 'Total_Notas',
    'Registros_Total', 'Registros_Promedio',
    'PVs_Total', 'PVs_Promedio',
    'Score_Global_Promedio', 'Originalidad_Promedio', 'Diversidad_Promedio'
]

resumen_cluster.to_csv('resumen_por_cluster.csv', index=False, encoding='utf-8-sig')
print("✅ Exportado: 'resumen_por_cluster.csv'")

# CSV 3: Top temas por editor
temas_export = []
for editor in metricas_editor['Editor']:
    top_temas = temas_por_editor[editor]
    for tema_data in top_temas:
        temas_export.append({
            'Editor': editor,
            'Tema': tema_data['tema'],
            'Num_Notas': tema_data['num_notas'],
            'Porcentaje': tema_data['porcentaje']
        })

df_temas_export = pd.DataFrame(temas_export)
df_temas_export.to_csv('temas_por_editor.csv', index=False, encoding='utf-8-sig')
print("✅ Exportado: 'temas_por_editor.csv'")


💾 FASE 9: EXPORTACIÓN DE RESULTADOS
✅ Exportado: 'resultados_por_editor.csv'
✅ Exportado: 'resumen_por_cluster.csv'
✅ Exportado: 'temas_por_editor.csv'


GENERAR TABLA RESUMEN PARA NEGOCIO

In [ ]:
print("\n" + "="*80)
print("📋 RESUMEN EJECUTIVO")
print("="*80)

print("\n🎯 OBJETIVO ALCANZADO:")
print("Se ha segmentado exitosamente a los editores de Milenio.com en 4 grupos")
print("de desempeño usando algoritmos de Machine Learning (KMeans).\n")

print("📊 RESULTADOS POR ETIQUETA:\n")

for idx, row in resumen_cluster.iterrows():
    print(f"{'='*70}")
    print(f"🏷️  {row['Etiqueta'].upper()}")
    print(f"{'='*70}")
    print(f"👥 Editores: {int(row['Num_Editores'])}")
    print(f"📝 Notas: {int(row['Total_Notas']):,}")
    print(f"🎯 Registros Totales: {int(row['Registros_Total']):,}")
    print(f"📊 Registros Promedio: {int(row['Registros_Promedio']):,}")
    print(f"👁️  PVs Totales: {int(row['PVs_Total']):,}")
    print(f"⭐ Score Global: {row['Score_Global_Promedio']:.1f}/100")
    print(f"✍️  Originalidad: {row['Originalidad_Promedio']*100:.1f}%")
    print(f"🎨 Diversidad Temática: {row['Diversidad_Promedio']:.2f}")
    print()

print("\n📈 MÉTRICAS CLAVE DEL MODELO:")
print(f"   • Clusters generados: {K_OPTIMO}")
print(f"   • Silhouette Score: {silhouette_scores[K_OPTIMO-2]:.3f}")
print(f"   • Varianza PCA explicada: {sum(pca.explained_variance_ratio_)*100:.1f}%")
print(f"   • Features utilizados: {len(features_para_clustering)}")

print("\n💡 PRÓXIMOS PASOS:")
print("   1. Revisar editores en 'Sapitos y Sapitas' para capacitación")
print("   2. Estudiar estrategias de 'Reyes y Reinas' para replicar")
print("   3. Analizar temas dominantes por cluster para asignación de notas")
print("   4. Monitorear índice de originalidad para fomentar contenido propio")

print("\n" + "="*80)
print("✅ ANÁLISIS COMPLETO")
print("="*80)
print("\n📁 ARCHIVOS GENERADOS:")
print("   • dashboard_editorial_milenio.html")
print("   • resultados_por_editor.csv")
print("   • resumen_por_cluster.csv")
print("   • temas_por_editor.csv")

print("\n🎉 ¡Listo para compartir con el equipo editorial!")
print("="*80)


📋 RESUMEN EJECUTIVO

🎯 OBJETIVO ALCANZADO:
Se ha segmentado exitosamente a los editores de Milenio.com en 4 grupos
de desempeño usando algoritmos de Machine Learning (KMeans).

📊 RESULTADOS POR ETIQUETA:

🏷️  DUQUES Y DUQUESAS
👥 Editores: 13
📝 Notas: 2,610
🎯 Registros Totales: 2,684
📊 Registros Promedio: 206
👁️  PVs Totales: 1,873,721
⭐ Score Global: 8.8/100
✍️  Originalidad: 21.2%
🎨 Diversidad Temática: 3.45

🏷️  PRÍNCIPES Y PRINCESAS
👥 Editores: 31
📝 Notas: 12,998
🎯 Registros Totales: 28,359
📊 Registros Promedio: 914
👁️  PVs Totales: 40,434,756
⭐ Score Global: 23.0/100
✍️  Originalidad: 21.0%
🎨 Diversidad Temática: 3.75

🏷️  REYES Y REINAS
👥 Editores: 13
📝 Notas: 11,993
🎯 Registros Totales: 49,141
📊 Registros Promedio: 3,780
👁️  PVs Totales: 55,682,178
⭐ Score Global: 39.2/100
✍️  Originalidad: 22.7%
🎨 Diversidad Temática: 4.25

🏷️  SAPITOS Y SAPITAS
👥 Editores: 60
📝 Notas: 18,211
🎯 Registros Totales: 8,448
📊 Registros Promedio: 140
👁️  PVs Totales: 20,931,114
⭐ Score Global: 7.6/10

DESCARGAR ARCHIVOS (en Google Colab)

In [ ]:
from google.colab import files

print("\n📥 Descargando archivos generados...")
files.download('dashboard_editorial_milenio.html')
files.download('resultados_por_editor.csv')
files.download('resumen_por_cluster.csv')
files.download('temas_por_editor.csv')

print("\n✅ ¡Descarga completa! Revisa tus archivos descargados.")


📥 Descargando archivos generados...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ ¡Descarga completa! Revisa tus archivos descargados.
